# C2 — Pipeline Supervisionado

## Objetivo

Construir a arquitetura do pipeline supervisionado **antes** da
comparação definitiva entre modelos.

O pipeline deve garantir:

- separação temporal (treino / validação / holdout);
- prevenção de leakage;
- tratamento consistente das variáveis;
- transformação das variáveis (numéricas e categóricas);
- reprodução do processamento;
- separação entre desenvolvimento e holdout final.

**Este notebook não treina modelo definitivo.** Ele só prepara a
arquitetura que C3+ vão usar.

## 1. Bootstrap — raiz do projeto e imports

In [3]:
from pathlib import Path
import sys

# Sobe diretórios até encontrar a pasta `src/` (raiz do projeto)
ROOT_DIR = Path.cwd().resolve()
while not (ROOT_DIR / "src").exists() and ROOT_DIR != ROOT_DIR.parent:
    ROOT_DIR = ROOT_DIR.parent

if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("Projeto:", ROOT_DIR)

Projeto: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador


In [4]:
import pandas as pd

from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run

from src.evaluation.temporal import TemporalSplit
from src.evaluation.errors import check_leakage
from src.models.pipeline import build_pipeline

env.carregar_dotenv()
env.garantir_diretorios()

print("Imports: OK")

Imports: OK


In [5]:
# Sanidade: mostra as configurações que vamos usar
print(C.resumo())

ROOT_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador
DATA_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data
RAW_DIR           = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\raw
PROCESSED_DIR     = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\processed
ANALYTICAL_DIR    = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\analytical
TREINO_DIR        = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\treino
OUTPUTS_DIR       = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs
RUNS_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\outputs\runs
LOGS_DIR          = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\logs
CONFIGS_DIR       = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\configs
SUPERVISED_YAML   = C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integra

## 2. Carregar dataset analítico

O C2 **consome** o dataset produzido pela etapa anterior — não
reprocessa os dados brutos.

In [6]:
ANALYTICAL_PATH = (
    ROOT_DIR
    / "data"
    / "analytical"
    / "dataset_supervisionado.parquet"
)

assert ANALYTICAL_PATH.exists(), (
    f"Dataset analítico não encontrado: {ANALYTICAL_PATH}\n"
    f"Gere-o na etapa anterior (Pessoa A) antes de rodar o C2."
)

df = pd.read_parquet(ANALYTICAL_PATH)
print("shape:", df.shape)
df.head()

AssertionError: Dataset analítico não encontrado: C:\Users\ivanI\OneDrive\Documentos\Projetos\Projeto_Integrador\data\analytical\dataset_supervisionado.parquet
Gere-o na etapa anterior (Pessoa A) antes de rodar o C2.

## 3. Definição do alvo, coluna temporal e features

Todos os nomes vêm do `supervised.yaml` — **nada hard-coded**.

In [ ]:
TARGET       = C.cfg("target", "coluna")
DATE_COLUMN  = C.cfg("coluna_tempo")
PROIBIDAS    = list(C.cfg("features", "proibidas") or [])
FEATURES     = C.features_finais()

print(f"TARGET        = {TARGET!r}")
print(f"DATE_COLUMN   = {DATE_COLUMN!r}")
print(f"FEATURES      = {len(FEATURES)} colunas")
print(f"PROIBIDAS     = {PROIBIDAS}")

assert TARGET is not None, "target.coluna ausente em supervised.yaml"
assert DATE_COLUMN is not None, "coluna_tempo ausente em supervised.yaml"
assert TARGET in df.columns, f"{TARGET!r} não está no dataset analítico"
assert DATE_COLUMN in df.columns, f"{DATE_COLUMN!r} não está no dataset analítico"

## 4. Normalizar a coluna temporal

Garantimos que `DATE_COLUMN` é `datetime` antes do split.

In [ ]:
df[DATE_COLUMN] = pd.to_datetime(df[DATE_COLUMN], errors="coerce")

n_na_data = int(df[DATE_COLUMN].isna().sum())
print(f"Datas inválidas (NaT): {n_na_data:,}")
print(f"Período: {df[DATE_COLUMN].min()} → {df[DATE_COLUMN].max()}")

## 5. Split temporal (com trava do holdout)

Usamos `TemporalSplit` — a classe expõe apenas `treino_validacao()`
durante a fase de seleção. O holdout só é liberado após
`split.concluir_selecao()` (chamada em C6/G8).

In [ ]:
split = TemporalSplit(
    df,
    coluna_tempo=DATE_COLUMN,
    frac_treino=float(C.cfg("temporal", "fracao", "treino") or 0.70),
    frac_validacao=float(C.cfg("temporal", "fracao", "validacao") or 0.15),
)

print(split.resumo())

In [ ]:
treino, validacao = split.treino_validacao()

print(f"treino    : {len(treino):,} linhas")
print(f"validacao : {len(validacao):,} linhas")
print(f"holdout   : {split.n_holdout:,} linhas (bloqueado)")

# Prova de que o holdout está bloqueado:
try:
    split.holdout()
except RuntimeError as e:
    print(f"\n[ok] holdout bloqueado, como esperado:")
    print(" ", str(e).splitlines()[0])

## 6. Separação X / y e checagem de leakage

Antes de montar o pipeline, verificamos que nenhuma coluna proibida
está em X. A lista proibida vem do `supervised.yaml`.

In [ ]:
X_train = treino.drop(columns=[TARGET])
y_train = treino[TARGET]

X_val   = validacao.drop(columns=[TARGET])
y_val   = validacao[TARGET]

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}    y_val  : {y_val.shape}")

In [ ]:
# Checagem de leakage — lê `features.proibidas` do supervised.yaml
check_leakage(X_train)
check_leakage(X_val)

## 7. Construir o pipeline

Nesta etapa só montamos o **pré-processamento** (imputação + escala +
one-hot). O classificador será injetado em C3+.

In [ ]:
pipeline = build_pipeline(X=X_train, y=y_train)
pipeline

In [ ]:
# Sanidade: fit_transform só no treino
X_train_proc = pipeline.fit_transform(X_train)
X_val_proc   = pipeline.transform(X_val)

print(f"X_train original : {X_train.shape}")
print(f"X_train processado: {X_train_proc.shape}")
print(f"X_val   processado: {X_val_proc.shape}")

## 8. Persistir o pipeline e o split

Salvamos o objeto do pipeline (não treinado) e o esquema de split para
que C3+ carreguem sem refazer a arquitetura.

In [ ]:
import joblib

OUT_DIR = C.OUTPUTS_DIR / "C2"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pipeline_path = OUT_DIR / "preprocessor.joblib"
joblib.dump(pipeline, pipeline_path)

print("Pipeline salvo em:", pipeline_path)

In [ ]:
# Registra os índices temporais do split para reuso
split_info = {
    "coluna_tempo":   DATE_COLUMN,
    "frac_treino":    split.frac_treino,
    "frac_validacao": split.frac_validacao,
    "n_total":        len(df),
    "n_treino":       split.n_treino,
    "n_validacao":    split.n_validacao,
    "n_holdout":      split.n_holdout,
    "periodo_treino": [
        str(treino[DATE_COLUMN].min()),
        str(treino[DATE_COLUMN].max()),
    ],
    "periodo_validacao": [
        str(validacao[DATE_COLUMN].min()),
        str(validacao[DATE_COLUMN].max()),
    ],
}

import json
(OUT_DIR / "split_info.json").write_text(
    json.dumps(split_info, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8",
)
print(json.dumps(split_info, indent=2, ensure_ascii=False))

## 9. Registrar execução

Gravamos `metadata.json` + `summary.txt` em `outputs/runs/<RUN_ID>/`
para garantir reprodutibilidade.

In [ ]:
with Run(modelo="C2_pipeline", notas="Arquitetura do pipeline supervisionado") as r:
    r.add_dataset(
        n_rows=len(df),
        train_rows=split.n_treino,
        valid_rows=split.n_validacao,
        test_rows=split.n_holdout,
    )
    r.features = {
        "n_features": len(FEATURES),
        "numericas":  list(C.cfg("features", "numericas") or []),
        "categoricas": list(C.cfg("features", "categoricas") or []),
    }
    r.parametros = {
        "target": TARGET,
        "coluna_tempo": DATE_COLUMN,
        "frac_treino": split.frac_treino,
        "frac_validacao": split.frac_validacao,
    }
    r.metrica(n_features_processadas=int(X_train_proc.shape[1]))

print("Run registrado em:", r.pasta)

## 10. Resultado esperado

Ao final deste notebook, você deve conseguir afirmar:

> *"Temos um pipeline que recebe os dados e realiza o processamento
> sem utilizar informações futuras ou do holdout."*

O holdout **permanece bloqueado** — só será liberado em G8, após
`split.concluir_selecao()`.

**Próximo passo:** C3 — baseline (`DummyClassifier`) para calibrar
expectativas.